# Notebook 01 — Data Loading & Quality Control

**Dataset:** GSE116256 — van Galen et al., Cell 2019  
**Goal:** Load raw single-cell expression data and filter low-quality cells.

## What is scRNA-seq?

Unlike bulk RNA-seq (which gives the *average* expression across thousands of cells), **single-cell RNA-seq (scRNA-seq)** measures gene expression in **each individual cell**. This allows us to:
- Identify distinct cell populations in a tumor
- Detect rare malignant subclones
- Reconstruct differentiation trajectories

The van Galen 2019 dataset contains **38,410 cells** from 16 AML patients and 5 healthy donors. Each cell has a measurement for ~33,000 genes.

## AnnData format

Scanpy uses the **AnnData** object to store scRNA-seq data:
```
adata
  .X         → expression matrix (cells × genes)
  .obs       → cell metadata (patient, cell type, QC metrics)
  .var       → gene metadata (gene names, HVG flag)
  .obsm      → embeddings (PCA, UMAP)
  .uns       → unstructured data (clustering results, PAGA graph)
```

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.utils import (
    load_all_samples,
    compute_qc_metrics,
    filter_cells_qc,
    plot_qc_violin,
    set_plot_style
)

set_plot_style()
sc.settings.verbosity = 1
sc.logging.print_header()
print(f"Scanpy version: {sc.__version__}")

## 1.1 — Load all samples

Each sample is one patient (AML or healthy bone marrow donor).  
We load all `.dem.txt.gz` + `.anno.txt.gz` pairs and concatenate them.

> **Before running:** Make sure you downloaded the data into `../data/raw/`  
> See `../data/README.md` for download instructions.

In [ ]:
DATA_DIR = "../data/raw"

adata = load_all_samples(DATA_DIR)
print(f"\nDataset shape: {adata.shape}")
print(f"Columns in .obs: {adata.obs.columns.tolist()}")

In [ ]:
# Overview of the AnnData object
print(adata)
adata.obs.head()

In [ ]:
# Sample composition
sample_counts = adata.obs['sample'].value_counts()
aml_samples = adata.obs[adata.obs['is_aml']]['sample'].unique()
bm_samples  = adata.obs[~adata.obs['is_aml']]['sample'].unique()

print(f"AML samples ({len(aml_samples)}): {sorted(aml_samples)}")
print(f"Healthy donors ({len(bm_samples)}): {sorted(bm_samples)}")
print(f"\nCells per sample:")
print(sample_counts)

## 1.2 — Quality Control

Three standard QC metrics for scRNA-seq:

| Metric | Description | Low quality if... |
|--------|-------------|-------------------|
| `n_genes_by_counts` | Number of detected genes per cell | < 200 → empty droplet |
| `total_counts` | Total UMI counts per cell | Too low → poor capture |
| `pct_counts_mt` | % mitochondrial reads | > 20% → dying/stressed cell |

Mitochondrial genes (MT-) are overrepresented in dying cells because cytoplasmic mRNA degrades faster than mitochondrial mRNA.

In [ ]:
adata = compute_qc_metrics(adata)
print("QC metrics computed.")
print(adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())

In [ ]:
plot_qc_violin(adata, save_path="../figures/01_qc_violin.png")

In [ ]:
# Scatter: total_counts vs n_genes — helps spot outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    adata.obs['total_counts'], adata.obs['n_genes_by_counts'],
    alpha=0.1, s=1, c='steelblue'
)
axes[0].set_xlabel('Total counts')
axes[0].set_ylabel('Genes detected')
axes[0].set_title('Counts vs Genes per cell')

axes[1].scatter(
    adata.obs['total_counts'], adata.obs['pct_counts_mt'],
    alpha=0.1, s=1, c='#c0392b'
)
axes[1].set_xlabel('Total counts')
axes[1].set_ylabel('% Mitochondrial')
axes[1].set_title('Counts vs % Mitochondrial')
axes[1].axhline(20, color='black', linestyle='--', linewidth=1, label='Threshold (20%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/01_qc_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 — Filter low-quality cells

In [ ]:
# Adjust thresholds based on the violin plots above
adata = filter_cells_qc(
    adata,
    min_genes=200,
    max_genes=6000,
    max_pct_mt=20.0
)

print(f"\nFinal dataset: {adata.n_obs} cells × {adata.n_vars} genes")

In [ ]:
# Save filtered object for next notebook
adata.write('../data/raw/adata_01_qc.h5ad')
print("Saved: ../data/raw/adata_01_qc.h5ad")

## Summary

| Step | Cells |
|------|-------|
| Before QC | (see above) |
| After QC | (see above) |

**Next:** Notebook 02 — Normalization, HVG selection, PCA, Harmony batch correction.